# Rental Rights Test Collection

This notebook builds the ground-truth question set used to evaluate the Victorian Rental Rights RAG system. Questions are linked to the knowledge-base chunks that contain the information needed to answer them.

In [1]:
from pathlib import Path
import pandas as pd

CURRENT_DIR = Path.cwd()

if (CURRENT_DIR / "data" / "processed").exists():
    DATA_DIR = CURRENT_DIR / "data" / "processed"
elif (CURRENT_DIR.parent / "data" / "processed").exists():
    DATA_DIR = CURRENT_DIR.parent / "data" / "processed"
else:
    raise FileNotFoundError("Could not locate data/processed folder")

kb_path = DATA_DIR / "rental_kb_chunks.jsonl"

kb_df = pd.read_json(kb_path, lines=True)

print("Knowledge-base chunks loaded:", len(kb_df))
display(kb_df.head())

Knowledge-base chunks loaded: 32


,chunk_id,document_id,document_name,page,text,source_file
0,RG_P04,RG,Renters Guide,4,Introduction\nVictoria has some of the stronge...,Renters Guide.pdf
1,RG_P07,RG,Renters Guide,7,Before you apply\nDocuments and information yo...,Renters Guide.pdf
2,RG_P08,RG,Renters Guide,8,More information: consumer.vic.gov.au/unlawful...,Renters Guide.pdf
3,RG_P09,RG,Renters Guide,9,Read through and complete the rental applicati...,Renters Guide.pdf
4,RG_P11,RG,Renters Guide,11,Communicating with your rental provider\nYou c...,Renters Guide.pdf


## 1. Review Knowledge-Base Coverage

Review the available chunks and their main topics before creating the evaluation questions.

In [2]:
# Create a compact overview of the available KB chunks

chunk_overview = kb_df[
    ["chunk_id", "document_name", "page", "text"]
].copy()

chunk_overview["preview"] = (
    chunk_overview["text"]
    .str.replace("\n", " ", regex=False)
    .str.slice(0, 180)
)

display(
    chunk_overview[
        ["chunk_id", "document_name", "page", "preview"]
    ]
)

,chunk_id,document_name,page,preview
0,RG_P04,Renters Guide,4,Introduction Victoria has some of the stronges...
1,RG_P07,Renters Guide,7,Before you apply Documents and information you...
2,RG_P08,Renters Guide,8,More information: consumer.vic.gov.au/unlawful...
3,RG_P09,Renters Guide,9,Read through and complete the rental applicati...
4,RG_P11,Renters Guide,11,Communicating with your rental provider You ca...
5,RG_P12,Renters Guide,12,Rental minimum standards guide There are 15 ca...
6,RG_P13,Renters Guide,13,Learn more about each of the minimum standards...
7,RG_P14,Renters Guide,14,Kitchen The property must have a kitchen with:...
8,RG_P15,Renters Guide,15,Locks The property’s external entry doors must...
9,RG_P16,Renters Guide,16,Ventilation Rental properties must have adequa...


## 2. Create Known Questions

Known questions have an answer that is directly available in one or more knowledge-base chunks. The first set covers a range of rental topics rather than focusing on one area.

In [3]:
# Initial Known question set

known_questions = [
    {
        "question_id": "K01",
        "question": "Can a rental provider ask me about disputes with a previous landlord when I apply?",
        "question_type": "Known",
        "relevant_chunk_ids": ["RG_P07"]
    },
    {
        "question_id": "K02",
        "question": "Can a rental provider accept an offer above the advertised rent?",
        "question_type": "Known",
        "relevant_chunk_ids": ["RG_P08"]
    },
    {
        "question_id": "K03",
        "question": "Does a Victorian rental property need to have a fixed heater?",
        "question_type": "Known",
        "relevant_chunk_ids": ["RG_P13", "MS_P01"]
    },
    {
        "question_id": "K04",
        "question": "How long do I have to return my completed condition report after moving in?",
        "question_type": "Known",
        "relevant_chunk_ids": ["RG_P17", "RG_P19"]
    },
    {
        "question_id": "K05",
        "question": "How much rent can I be asked to pay in advance?",
        "question_type": "Known",
        "relevant_chunk_ids": ["RG_P21"]
    },
    {
        "question_id": "K06",
        "question": "How much notice must I receive before my rent is increased?",
        "question_type": "Known",
        "relevant_chunk_ids": ["RG_P22"]
    },
    {
        "question_id": "K07",
        "question": "Is a broken toilet considered an urgent repair?",
        "question_type": "Known",
        "relevant_chunk_ids": ["RG_P24"]
    },
    {
        "question_id": "K08",
        "question": "What happens if my rental provider wants to refuse my request to keep a pet?",
        "question_type": "Known",
        "relevant_chunk_ids": ["RG_P28"]
    }
]

test_df = pd.DataFrame(known_questions)

display(test_df)

,question_id,question,question_type,relevant_chunk_ids
0,K01,Can a rental provider ask me about disputes wi...,Known,[RG_P07]
1,K02,Can a rental provider accept an offer above th...,Known,[RG_P08]
2,K03,Does a Victorian rental property need to have ...,Known,"[RG_P13, MS_P01]"
3,K04,How long do I have to return my completed cond...,Known,"[RG_P17, RG_P19]"
4,K05,How much rent can I be asked to pay in advance?,Known,[RG_P21]
5,K06,How much notice must I receive before my rent ...,Known,[RG_P22]
6,K07,Is a broken toilet considered an urgent repair?,Known,[RG_P24]
7,K08,What happens if my rental provider wants to re...,Known,[RG_P28]


## 3. Add Expected Answers

Record a concise ground-truth answer for each question based only on the information contained in the knowledge base.

In [4]:
# Add expected answers for the Known questions

expected_answers = {
    "K01": (
        "No. A rental provider or agent cannot ask whether you have taken legal action "
        "or had a dispute with a previous rental provider."
    ),
    "K02": (
        "No. Rental providers and agents cannot ask for, invite or accept offers of rent "
        "higher than the advertised price."
    ),
    "K03": (
        "Yes. Rental properties must have a fixed heater in good working order in the main "
        "living area. For rental agreements starting from 29 March 2023, the heater must "
        "also be energy efficient."
    ),
    "K04": (
        "You must return one completed and signed copy of the condition report within "
        "5 business days of moving in."
    ),
    "K05": (
        "If rent is paid weekly, you can be asked for up to 2 weeks' rent in advance. "
        "If rent is paid monthly and the weekly rent is $900 or less, the maximum is one "
        "month's rent. Different rules apply where the weekly rent is above $900."
    ),
    "K06": (
        "The rental provider must give you a Notice of rent increase at least 90 days "
        "before the rent increase takes effect."
    ),
    "K07": (
        "Yes. A blocked or broken toilet system is legally defined as an urgent repair "
        "and must be repaired immediately."
    ),
    "K08": (
        "If the rental provider wants to refuse consent for a pet, they must apply to "
        "VCAT within 14 days. VCAT then decides whether refusing consent is reasonable."
    )
}

test_df["expected_answer"] = test_df["question_id"].map(expected_answers)

display(test_df)

,question_id,question,question_type,relevant_chunk_ids,expected_answer
0,K01,Can a rental provider ask me about disputes wi...,Known,[RG_P07],No. A rental provider or agent cannot ask whet...
1,K02,Can a rental provider accept an offer above th...,Known,[RG_P08],No. Rental providers and agents cannot ask for...
2,K03,Does a Victorian rental property need to have ...,Known,"[RG_P13, MS_P01]",Yes. Rental properties must have a fixed heate...
3,K04,How long do I have to return my completed cond...,Known,"[RG_P17, RG_P19]",You must return one completed and signed copy ...
4,K05,How much rent can I be asked to pay in advance?,Known,[RG_P21],"If rent is paid weekly, you can be asked for u..."
5,K06,How much notice must I receive before my rent ...,Known,[RG_P22],The rental provider must give you a Notice of ...
6,K07,Is a broken toilet considered an urgent repair?,Known,[RG_P24],Yes. A blocked or broken toilet system is lega...
7,K08,What happens if my rental provider wants to re...,Known,[RG_P28],If the rental provider wants to refuse consent...


## 4. Create Inferred Questions

Inferred questions require information from multiple knowledge-base chunks to produce a complete answer.

In [5]:
# Initial Inferred question set

inferred_questions = [
    {
        "question_id": "I01",
        "question": "My rental has no working fixed heater. Is this considered an urgent repair, and what can I do if the rental provider does not respond?",
        "question_type": "Inferred",
        "relevant_chunk_ids": ["RG_P13", "MS_P01", "RG_P24", "RG_P25"]
    },
    {
        "question_id": "I02",
        "question": "My rent is being increased and I think the new amount is too high. How much notice should I receive and what can I do to challenge it?",
        "question_type": "Inferred",
        "relevant_chunk_ids": ["RG_P22", "RG_P23"]
    },
    {
        "question_id": "I03",
        "question": "My rental provider wants to claim my bond for damage that was already there when I moved in. What information could help me dispute the claim?",
        "question_type": "Inferred",
        "relevant_chunk_ids": ["RG_P17", "RG_P34", "RG_P35"]
    },
    {
        "question_id": "I04",
        "question": "Can my rental provider evict me because I have tried to exercise my rental rights, and what do they normally need to end the agreement?",
        "question_type": "Inferred",
        "relevant_chunk_ids": ["RG_P32", "RG_P33"]
    }
]

inferred_df = pd.DataFrame(inferred_questions)

display(inferred_df)

,question_id,question,question_type,relevant_chunk_ids
0,I01,My rental has no working fixed heater. Is this...,Inferred,"[RG_P13, MS_P01, RG_P24, RG_P25]"
1,I02,My rent is being increased and I think the new...,Inferred,"[RG_P22, RG_P23]"
2,I03,My rental provider wants to claim my bond for ...,Inferred,"[RG_P17, RG_P34, RG_P35]"
3,I04,Can my rental provider evict me because I have...,Inferred,"[RG_P32, RG_P33]"


## 5. Add Expected Answers for Inferred Questions

Create ground-truth answers by combining the information contained across the relevant knowledge-base chunks.

In [6]:
# Add expected answers for the Inferred questions

inferred_answers = {
    "I01": (
        "Yes. A rental property must have a working fixed heater in the main living area, "
        "and a property that does not meet the minimum standards is considered an urgent repair. "
        "The rental provider must arrange the repair immediately. If they do not respond, you can "
        "organise and pay for the urgent repair yourself if it costs no more than $2500. The rental "
        "provider must reimburse you within 7 days, and if they do not, you can apply to RDRV."
    ),
    "I02": (
        "You must receive a Notice of rent increase at least 90 days before the increase takes effect. "
        "If you believe the increase is too high, you can ask Consumer Affairs Victoria for a free rent "
        "assessment within 30 days of receiving the notice. If you still cannot agree on the rent after "
        "the assessment, you can apply to RDRV or VCAT."
    ),
    "I03": (
        "Your condition report can help show that the damage was already present when you moved in, "
        "especially if you recorded the damage and took photos. The exit condition report can then be "
        "compared with the original condition report. A rental provider can claim the bond for damage "
        "caused by you or your visitors, but not for fair wear and tear. If you disagree with the bond "
        "claim, you can dispute it through RDRV."
    ),
    "I04": (
        "No. A rental provider cannot evict you because you have used or intend to use your rental rights. "
        "To end the rental agreement, they generally need a valid reason, the correct written Notice to "
        "vacate, and the required amount of notice. The notice period depends on the reason, although the "
        "guide states that in most instances it is 90 days."
    )
}

inferred_df["expected_answer"] = inferred_df["question_id"].map(inferred_answers)

display(inferred_df)

,question_id,question,question_type,relevant_chunk_ids,expected_answer
0,I01,My rental has no working fixed heater. Is this...,Inferred,"[RG_P13, MS_P01, RG_P24, RG_P25]",Yes. A rental property must have a working fix...
1,I02,My rent is being increased and I think the new...,Inferred,"[RG_P22, RG_P23]",You must receive a Notice of rent increase at ...
2,I03,My rental provider wants to claim my bond for ...,Inferred,"[RG_P17, RG_P34, RG_P35]",Your condition report can help show that the d...
3,I04,Can my rental provider evict me because I have...,Inferred,"[RG_P32, RG_P33]",No. A rental provider cannot evict you because...


## 6. Create Out-of-KB Questions

Out-of-KB questions test whether the RAG system can recognise when the available knowledge base does not contain enough information to provide a supported answer.

In [7]:
# Initial Out-of-KB question set

out_of_kb_questions = [
    {
        "question_id": "O01",
        "question": "Can I sublet my rental property to another person without my rental provider's permission?",
        "question_type": "Out-of-KB",
        "relevant_chunk_ids": [],
        "expected_answer": "The current knowledge base does not contain enough information to answer this question."
    },
    {
        "question_id": "O02",
        "question": "How much does it cost to make a rental application to VCAT?",
        "question_type": "Out-of-KB",
        "relevant_chunk_ids": [],
        "expected_answer": "The current knowledge base does not contain enough information to answer this question."
    },
    {
        "question_id": "O03",
        "question": "Can my rental provider install security cameras inside my rental property?",
        "question_type": "Out-of-KB",
        "relevant_chunk_ids": [],
        "expected_answer": "The current knowledge base does not contain enough information to answer this question."
    }
]

out_of_kb_df = pd.DataFrame(out_of_kb_questions)

display(out_of_kb_df)

,question_id,question,question_type,relevant_chunk_ids,expected_answer
0,O01,Can I sublet my rental property to another per...,Out-of-KB,[],The current knowledge base does not contain en...
1,O02,How much does it cost to make a rental applica...,Out-of-KB,[],The current knowledge base does not contain en...
2,O03,Can my rental provider install security camera...,Out-of-KB,[],The current knowledge base does not contain en...


## 7. Finalise the Version 1 Test Collection

Combine the Known, Inferred and Out-of-KB questions, check that all labelled relevant chunks exist in the knowledge base, and save the test collection for later evaluation.

In [8]:
# Combine the three question types

test_collection_df = pd.concat(
    [test_df, inferred_df, out_of_kb_df],
    ignore_index=True
)

# Check that all labelled relevant chunks exist in the KB
valid_chunk_ids = set(kb_df["chunk_id"])

invalid_chunks = []

for _, row in test_collection_df.iterrows():
    for chunk_id in row["relevant_chunk_ids"]:
        if chunk_id not in valid_chunk_ids:
            invalid_chunks.append((row["question_id"], chunk_id))

print("Total questions:", len(test_collection_df))
print("\nQuestion types:")
print(test_collection_df["question_type"].value_counts())

print("\nInvalid relevant chunk IDs:", invalid_chunks)

display(test_collection_df)

Total questions: 15

Question types:
question_type
Known        8
Inferred     4
Out-of-KB    3
Name: count, dtype: int64

Invalid relevant chunk IDs: []


,question_id,question,question_type,relevant_chunk_ids,expected_answer
0,K01,Can a rental provider ask me about disputes wi...,Known,[RG_P07],No. A rental provider or agent cannot ask whet...
1,K02,Can a rental provider accept an offer above th...,Known,[RG_P08],No. Rental providers and agents cannot ask for...
2,K03,Does a Victorian rental property need to have ...,Known,"[RG_P13, MS_P01]",Yes. Rental properties must have a fixed heate...
3,K04,How long do I have to return my completed cond...,Known,"[RG_P17, RG_P19]",You must return one completed and signed copy ...
4,K05,How much rent can I be asked to pay in advance?,Known,[RG_P21],"If rent is paid weekly, you can be asked for u..."
5,K06,How much notice must I receive before my rent ...,Known,[RG_P22],The rental provider must give you a Notice of ...
6,K07,Is a broken toilet considered an urgent repair?,Known,[RG_P24],Yes. A blocked or broken toilet system is lega...
7,K08,What happens if my rental provider wants to re...,Known,[RG_P28],If the rental provider wants to refuse consent...
8,I01,My rental has no working fixed heater. Is this...,Inferred,"[RG_P13, MS_P01, RG_P24, RG_P25]",Yes. A rental property must have a working fix...
9,I02,My rent is being increased and I think the new...,Inferred,"[RG_P22, RG_P23]",You must receive a Notice of rent increase at ...


In [9]:
# Save the Version 1 test collection

test_output_path = DATA_DIR / "rental_test_collection_v1.jsonl"

test_collection_df.to_json(
    test_output_path,
    orient="records",
    lines=True,
    force_ascii=False
)

print("Saved test collection to:")
print(test_output_path)

print("\nQuestions saved:", len(test_collection_df))

Saved test collection to:
/Users/seanrichards/Documents/University/DS Case Studies/Project/data/processed/rental_test_collection_v1.jsonl

Questions saved: 15
